In [ ]:
import sys

from pathlib import Path

sys.path.append(str(Path.cwd().parent/'src'))

In [ ]:
from pdf_ingestion import PdfIngestion

In [ ]:
ingestor = PdfIngestion(chunk_size=1200,chunk_overlap=180)

pdf_chunks = ingestor.process('C:\Projets_rag_personnels\data\pdf_files\Capstone_FinalReport.pdf')


print(f'Total of chunks : {len(pdf_chunks)}')

for i, chunk in enumerate(pdf_chunks[18:21]):

  print(f'CHUNK {i+1}')
  print(chunk.page_content)
  print(f'Source :page {chunk.metadata.get('page')}')

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv
import os

load_dotenv()
print(os.getenv('OPENAI_API_KEY')[:10] + '...')





embed = OpenAIEmbeddings(model='text-embedding-3-small')

pdf_vector_store = Chroma(collection_name='Capstone_focused_docs',
                          embedding_function=embed,
                          persist_directory='/content/drive/MyDrive/Cour de RAG/mes _projets RAG/vector_store_new')

pdf_vector_store.add_documents(pdf_chunks)

retriever = pdf_vector_store.as_retriever(search_kwargs = {'k':3})


# let us test our retriever

sample_search = retriever.invoke("tell me about the analysis of revenue concentration")

for i, result in enumerate(sample_search[:3]):

  print(f'RESULT {i+1}')

  print(result.page_content,'\n')

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser


llm = ChatOpenAI(model = 'gpt-4o-mini')

parser = StrOutputParser()

prompt = ChatPromptTemplate.from_template ("""
                                           
You are a technical assistant for our data analytics team.
Answer the question below focusing on the context below.
If there is no answer in the context, just say: "there is no answer"


QUESTION:
{question}


CONTEXT:
{context}


ANSWER:
Be precise and very concise.

""")

rag_chain = (

{'context': lambda x: retriever.invoke(x['question']),
 'question': lambda x: x['question']}

|prompt
|llm
|parser


)



In [ ]:
question = 'what is the sector analysis tries to figure out?'

question

In [ ]:
response = rag_chain.invoke({'question':question})

In [ ]:
print(f'USER_Query : {question}\n')

print('*'*50, '\n')

print(f'RAG_Answer : \n  {response}' )

